# 02 - Uji Preprocessing Teks (S1 sampai S4)

**Tujuan:** menguji modul `src/preprocess.py` pada kalimat contoh, kasus khusus dari EDA, dan data sebenarnya, serta membandingkan ukuran kosakata tiap skenario.

| Skenario | Perlakuan |
|----------|-----------|
| S1 | Teks mentah (baseline) |
| S2 | S1 + huruf kecil, hapus apostrof, hapus aksen, hapus angka dan tanda baca, buang token 1 karakter |
| S3 | S2 + stopword removal |
| S4 | S3 + stemming (PorterStemmer) |

In [1]:
import sys
import pandas as pd

sys.path.append('../src')
from preprocess import preprocess_text

skenario = ['S1', 'S2', 'S3', 'S4']

## 1. Kalimat Contoh

In [2]:
teks_uji = "Bond's new car is chasing 2 aliens in the streets!"
print('Teks asli:', teks_uji)
for s in skenario:
    print(s, '->', preprocess_text(teks_uji, s))

Teks asli: Bond's new car is chasing 2 aliens in the streets!
S1 -> Bond's new car is chasing 2 aliens in the streets!
S2 -> bonds new car is chasing aliens in the streets
S3 -> bonds new car chasing aliens streets
S4 -> bond new car chase alien street


## 2. Kasus Khusus dari EDA

Kasus ini muncul pada data: huruf beraksen, angka bersama huruf (22nd, 1990s), tanda hubung panjang, dan teks kosong.

In [3]:
kasus = [
    "Her fianc\u00e9 joins the 22nd century Pok\u00e9mon quest",
    "A mid\u2013century drama of the 1990s and the 20th Century Fox",
    "A man is on a quest, 3 D-days",
    None,
]
for teks in kasus:
    print('Asli:', teks)
    for s in ['S2', 'S3', 'S4']:
        print('  ', s, '->', repr(preprocess_text(teks, s)))
    print()

Asli: Her fiancé joins the 22nd century Pokémon quest
   S2 -> 'her fiance joins the century pokemon quest'
   S3 -> 'fiance joins century pokemon quest'
   S4 -> 'fianc join centuri pokemon quest'

Asli: A mid–century drama of the 1990s and the 20th Century Fox
   S2 -> 'mid century drama of the and the century fox'
   S3 -> 'mid century drama century fox'
   S4 -> 'mid centuri drama centuri fox'

Asli: A man is on a quest, 3 D-days
   S2 -> 'man is on quest days'
   S3 -> 'man quest days'
   S4 -> 'man quest day'

Asli: None
   S2 -> ''
   S3 -> ''
   S4 -> ''



Skenario yang tidak dikenal harus ditolak, bukan diproses diam-diam.

In [4]:
try:
    preprocess_text("contoh", 'S9')
except ValueError as e:
    print('Error yang diharapkan:', e)

Error yang diharapkan: scenario harus S1, S2, S3, atau S4


## 3. Uji pada Data Sebenarnya

In [5]:
df = pd.read_csv('../dataset/movies_clean.csv')
contoh = df['overview'].iloc[0]

print('Judul:', df['title'].iloc[0])
print()
print('Sinopsis asli:')
print(contoh)
for s in skenario:
    print()
    print(s + ':')
    print(preprocess_text(contoh, s))

Judul: Avatar

Sinopsis asli:
In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.

S1:
In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.

S2:
in the century paraplegic marine is dispatched to the moon pandora on unique mission but becomes torn between following orders and protecting an alien civilization

S3:
century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization

S4:
centuri parapleg marin dispatch moon pandora uniqu mission becom torn follow order protect alien civil


## 4. Ukuran Kosakata per Skenario (500 Sinopsis)

In [6]:
baris = []
for s in skenario:
    hasil = df['overview'].apply(lambda t: preprocess_text(t, s))
    token = hasil.str.split()
    kosakata = set(w for daftar in token for w in daftar)
    baris.append([s, len(kosakata), round(token.str.len().mean(), 1), int((token.str.len() == 0).sum())])

tabel = pd.DataFrame(baris, columns=['skenario', 'kosakata_unik', 'rata2_token_per_sinopsis', 'dokumen_kosong'])
tabel

,skenario,kosakata_unik,rata2_token_per_sinopsis,dokumen_kosong
0,S1,8181,54.9,0
1,S2,6101,53.1,0
2,S3,5988,31.9,0
3,S4,4616,31.9,0


## Kesimpulan

- Fungsi `preprocess_text` bekerja untuk keempat skenario, dan skenario yang salah ditolak dengan pesan error.
- Apostrof dihapus tanpa menyisakan token `s`, huruf beraksen diubah ke huruf biasa (fiancé menjadi fiance), dan token yang mengandung angka (22nd, 1990s, 20th) dibuang seluruhnya agar tidak tersisa token sampah seperti `nd` atau `th`.
- Kosakata unik turun bertahap: **8.181** (S1), **6.101** (S2), **5.988** (S3), dan **4.616** (S4). Stopword removal memangkas rata-rata token per sinopsis dari 53,1 menjadi 31,9, sedangkan stemming terutama menurunkan jumlah kosakata unik.
- Tidak ada dokumen yang menjadi kosong pada keempat skenario, sehingga semua 500 film tetap bisa dicari.

**Catatan untuk Tahap 4:** pada S1, `TfidfVectorizer` harus dibuat dengan `lowercase=False` dan `token_pattern=r"\S+"` agar benar-benar mentah. Pada S2 sampai S4, teks sudah dibersihkan sebelum masuk vectorizer.